[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_7_deep_learning_6d_pose/20_efficientpose.ipynb)

# Notebook 20 — EfficientPose: End-to-End 6D Pose from a Single Network

**Part 7: Deep Learning for 6D Pose** | Estimated time: 55 min

---

EfficientPose represents the next step beyond Objectron:  
instead of predicting bounding box corners, it predicts **rvec + tvec directly** for each detected object.

```
Classical pipeline:         EfficientPose:

  Detector (network)          One network does it all:
      ↓                         ┌── Class Net → class label
  Keypoints / corners           ├── Box Net → 2D bounding box
      ↓                         ├── Rotation Net → rvec
  PnP solver                    └── Translation Net → tvec
      ↓                              ↑
  rvec, tvec              EfficientNet backbone extracts features
```

> ⚠️ **Environment Note:** EfficientPose was implemented in **TensorFlow 1.x**.  
> Running it requires a special environment. This notebook teaches the architecture and  
> how to run inference — a Docker-based setup is the recommended approach.

## Recommended Watch

> Watch this **before** opening the notebook — focus on the EfficientPose section (first half of the video).

| Title | Link | Duration | Note |
|---|---|---|---|
| **6D Pose Estimation WITHOUT MARKERS for 3D Object Detection via FoundationPose & EfficientPose** | [▶ Watch](https://www.youtube.com/watch?v=mlXs5kIQ5p4) | ~20 min | Watch the EfficientPose section (first half) |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install numpy matplotlib opencv-python -q
    print('Running in Google Colab')
    print('Note: EfficientPose requires TF1 — install separately in a dedicated env')
else:
    print('Running locally')
    print('EfficientPose environment: Python 3.7 + TensorFlow 1.14 + CUDA 10.0')

## Learning Objectives

By the end of this notebook you will be able to:

- Describe the EfficientPose architecture (backbone + four heads)
- Explain why end-to-end learning is faster than detection + PnP
- Understand the EfficientDet anchor-based detection framework
- Set up the EfficientPose environment correctly
- Run inference and interpret `rotation`, `translation` outputs
- Know the LineMOD dataset and its object classes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

print('Imports ready.')

## 1. EfficientPose Architecture

EfficientPose builds on **EfficientDet** (an object detector) by adding two extra network heads:

```
                    EfficientNet Backbone
                    (feature pyramid: P3, P4, P5, P6, P7)
                         │
              ┌──────────┼──────────┬──────────┐
              │          │          │          │
          Class Net   Box Net  Rotation Net  Translation Net
              │          │          │          │
          class label  2D bbox   rvec (3D)  tvec (3D)
```

### The Rotation and Translation Subnetworks

Both rotation and translation nets use **iterative refinement**:

1. **Initial prediction** — coarse estimate from anchor features
2. **Iterative refinement** — stacked conv layers progressively refine
3. **Final output** — one rvec + tvec per detected anchor

This is the key difference from older methods:

| Old approach | EfficientPose approach |
|---|---|
| Find 2D keypoints | Predict rotation directly |
| Solve PnP | Predict translation directly |
| Two separate steps | One forward pass |
| Geometry-based | Learned end-to-end |

### Why EfficientNet Backbone?

EfficientNet uses **compound scaling** — simultaneously scaling network width, depth, and input resolution. This gives the best accuracy/speed trade-off. EfficientPose comes in 6 sizes (Φ=0 to 5), larger = more accurate but slower.

In [ ]:
# Visualize EfficientPose scaling across Phi variants
phi_configs = [
    {'phi': 0, 'input': 512,  'params_M': 3.9,  'fps_gpu': 32, 'acc': 55},
    {'phi': 1, 'input': 640,  'params_M': 6.6,  'fps_gpu': 28, 'acc': 61},
    {'phi': 2, 'input': 768,  'params_M': 8.1,  'fps_gpu': 22, 'acc': 65},
    {'phi': 3, 'input': 896,  'params_M': 12.0, 'fps_gpu': 16, 'acc': 69},
    {'phi': 4, 'input': 1024, 'params_M': 20.7, 'fps_gpu': 10, 'acc': 73},
    {'phi': 5, 'input': 1280, 'params_M': 33.7, 'fps_gpu': 6,  'acc': 76},
]

phis = [c['phi'] for c in phi_configs]
fps  = [c['fps_gpu'] for c in phi_configs]
acc  = [c['acc'] for c in phi_configs]
params = [c['params_M'] for c in phi_configs]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(phis, fps, color='steelblue', alpha=0.8)
axes[0].set_xlabel('EfficientPose Phi (Φ)', fontsize=11)
axes[0].set_ylabel('FPS (RTX GPU)', fontsize=11)
axes[0].set_title('Inference Speed vs Variant', fontsize=11)
axes[0].set_xticks(phis)
for i, v in enumerate(fps):
    axes[0].text(i, v+0.5, f'{v}', ha='center', fontsize=9)

axes[1].bar(phis, acc, color='coral', alpha=0.8)
axes[1].set_xlabel('EfficientPose Phi (Φ)', fontsize=11)
axes[1].set_ylabel('ADD-0.1d Accuracy (%)', fontsize=11)
axes[1].set_title('Pose Accuracy vs Variant', fontsize=11)
axes[1].set_xticks(phis)
for i, v in enumerate(acc):
    axes[1].text(i, v+0.5, f'{v}%', ha='center', fontsize=9)

axes[2].scatter(fps, acc, c=phis, cmap='viridis', s=200, zorder=5)
for i, cfg in enumerate(phi_configs):
    axes[2].annotate(f'Φ={cfg["phi"]}', (fps[i], acc[i]),
                     textcoords='offset points', xytext=(8, 0), fontsize=9)
axes[2].set_xlabel('FPS', fontsize=11)
axes[2].set_ylabel('Accuracy (%)', fontsize=11)
axes[2].set_title('Accuracy-Speed Trade-off', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.suptitle('EfficientPose: Choose Phi based on hardware and speed requirements',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Recommendation:')
print('  Phi=0: embedded systems, real-time (32 FPS)')
print('  Phi=2: balance (22 FPS, 65% accuracy)')
print('  Phi=5: max accuracy research (6 FPS, 76%)')

## 2. LineMOD Dataset — What EfficientPose Was Trained On

EfficientPose (in its original implementation) was trained and evaluated on **LineMOD** — a standard 6D pose benchmark.

```
LineMOD contains 13 objects:

  Ape       Benchvise  Cam      Can       Cat
  Driller   Duck       Eggbox   Glue      Holepuncher
  Iron      Lamp       Phone
```

### ADD Metric (Average Distance)

The standard accuracy metric for 6D pose:

$$\text{ADD-d} = \frac{1}{|M|} \sum_{x \in M} \|\hat{R}x + \hat{t} - (Rx + t)\|$$

Where:
- $M$ = set of 3D model points
- $\hat{R}, \hat{t}$ = predicted pose
- $R, t$ = ground truth pose

A prediction is **correct** if ADD < 10% of the object's diameter (ADD-0.1d).

In [ ]:
# Implement the ADD metric
def compute_ADD(model_pts, R_pred, t_pred, R_gt, t_gt):
    """
    Compute ADD (Average Distance) between predicted and GT pose.
    
    Args:
        model_pts: (N, 3) 3D model points
        R_pred:    (3,3) predicted rotation
        t_pred:    (3,) predicted translation
        R_gt:      (3,3) ground truth rotation
        t_gt:      (3,) ground truth translation
    
    Returns:
        add:        float, mean point distance
        is_correct: bool, True if add < 10% of object diameter
    """
    pts_pred = (R_pred @ model_pts.T).T + t_pred
    pts_gt   = (R_gt   @ model_pts.T).T + t_gt
    
    distances = np.linalg.norm(pts_pred - pts_gt, axis=1)
    add = distances.mean()
    
    # Object diameter (max distance between any two model points)
    from itertools import combinations
    dists = []
    # Sample 100 pairs for speed
    sample_idx = np.random.choice(len(model_pts), min(50, len(model_pts)), replace=False)
    for i in range(len(sample_idx)):
        for j in range(i+1, len(sample_idx)):
            d = np.linalg.norm(model_pts[sample_idx[i]] - model_pts[sample_idx[j]])
            dists.append(d)
    diameter = max(dists) if dists else 0.1
    
    is_correct = add < 0.1 * diameter
    return add, is_correct, diameter

# Example: generate a simple box-shaped model
np.random.seed(42)
r = 0.03
model_pts = np.random.uniform(-r, r, (100, 3))

# Ground truth pose
R_gt = np.eye(3)
t_gt = np.array([0.1, 0.05, 0.5])

print('=== ADD Metric Demo ===')
print()

# Test different prediction qualities
for rot_error_deg, trans_error_cm in [(0, 0), (5, 0.5), (15, 2.0), (30, 5.0)]:
    # Add small rotation error
    angle = np.radians(rot_error_deg)
    R_err = np.array([[np.cos(angle), -np.sin(angle), 0],
                      [np.sin(angle),  np.cos(angle), 0],
                      [0, 0, 1]])
    R_pred = R_err @ R_gt
    t_pred = t_gt + np.array([trans_error_cm/100, 0, 0])
    
    add, correct, diameter = compute_ADD(model_pts, R_pred, t_pred, R_gt, t_gt)
    print(f'Rotation error: {rot_error_deg:3d}° | Trans error: {trans_error_cm:.1f}cm | '
          f'ADD: {add*100:.2f}cm | Correct: {"✅" if correct else "❌"} '
          f'(threshold: {diameter*10:.2f}cm = 10% of {diameter*100:.1f}cm)')

---

## 🐳 New to Docker? Read This First

> **Docker** is a tool that packages software into a self-contained "container" — think of it as a mini-computer running *inside* your computer, with its own version of Python, its own libraries, and its own operating system. Nothing inside the container affects your main system, and nothing on your main system interferes with the container.

**Why do we need Docker for EfficientPose?**

EfficientPose was built with **TensorFlow 1.14**, which was released in 2019. Modern machines run Python 3.10+ and TensorFlow 2.x — and TF 1.x is incompatible with both. Trying to install TF 1.x directly on a modern machine almost always breaks other tools. Docker solves this cleanly by keeping the old environment completely isolated:

```
Your computer  (Python 3.11, TensorFlow 2.x, modern packages)
└── Docker container  (isolated, like a separate machine)
    ├── Python 3.7
    ├── TensorFlow 1.14
    ├── CUDA 10.0
    └── EfficientPose  ← runs perfectly in here
```

**Key Docker vocabulary — the only 4 terms you need:**

| Term | Plain-English meaning |
|------|----------------------|
| **Image** | A snapshot / blueprint of a complete environment. Downloaded once with `docker pull`. |
| **Container** | A running instance of an image. Started with `docker run`. Like booting up a mini-computer. |
| **`-v` flag** | Shares a folder between your real computer and the container so you can pass files in/out. |
| **`-it` flag** | Opens an interactive terminal inside the container (like SSH-ing into the mini-computer). |

**How to install Docker (one-time setup):**

1. Go to **https://www.docker.com/products/docker-desktop** and download Docker Desktop for your OS
2. Install and launch it — wait until the whale icon in your taskbar says "Docker Desktop is running"
3. That's it. The `docker pull` and `docker run` commands in Section 3 below do the rest automatically

> **No GPU or don't want to install Docker right now?** That's completely fine. Read through Section 3 to understand the workflow, then skip to Section 4 to learn how to interpret EfficientPose output. You can always come back to the setup later.

---

## 3. Setting Up EfficientPose

EfficientPose requires a specific environment. The most reliable approach is **Docker**.

### Option A: Docker (Recommended)

```bash
# Pull the TF1 GPU image
docker pull tensorflow/tensorflow:1.14.0-gpu-py3

# Clone EfficientPose
git clone https://github.com/ybkscht/EfficientPose.git
cd EfficientPose

# Run Docker with GPU + webcam access
docker run --gpus all -it \
    --device /dev/video0:/dev/video0 \
    -v $(pwd):/workspace \
    tensorflow/tensorflow:1.14.0-gpu-py3 \
    /bin/bash

# Inside container:
pip install opencv-python keras==2.2.5
python demo.py --phi 0 --weights weights/phi0linemod_best_ADD.h5 \
               --class_name cup --score_threshold 0.5
```

### Option B: Conda Environment (Windows/Linux)

```bash
conda create -n efficientpose python=3.7
conda activate efficientpose
pip install tensorflow-gpu==1.14.0
pip install keras==2.2.5 opencv-python
# Install CUDA 10.0 + cuDNN 7.6 manually from NVIDIA
```

---

### Downloading Weights

> ⬇️ **You must download the weights manually before running EfficientPose.** The model code is in the repo, but the trained weight files are large (~50–200 MB each) and are distributed separately via GitHub Releases.

**Step 1 — Go to the releases page:**

👉 https://github.com/ybkscht/EfficientPose/releases

**Step 2 — Download the weight file(s) you need:**

| File | Phi variant | Speed | Accuracy | Size |
|------|-------------|-------|----------|------|
| `phi0linemod_best_ADD.h5` | Φ=0 | Fastest (32 FPS) | 55% ADD | ~50 MB |
| `phi2linemod_best_ADD.h5` | Φ=2 | Balanced (22 FPS) | 65% ADD | ~90 MB |
| `phi5linemod_best_ADD.h5` | Φ=5 | Most accurate (6 FPS) | 76% ADD | ~200 MB |

**Start with Φ=0** (`phi0linemod_best_ADD.h5`) — it's the fastest and smallest. Only download larger variants if you specifically need higher accuracy.

**Step 3 — Place the file in the `weights/` folder inside the cloned repo:**

```
EfficientPose/
├── model.py
├── demo.py
├── weights/              ← create this folder if it doesn't exist
│   └── phi0linemod_best_ADD.h5   ← put the downloaded file here
└── ...
```

```bash
# Create the weights folder (inside the EfficientPose repo):
mkdir -p weights

# Then move your downloaded file into it:
mv ~/Downloads/phi0linemod_best_ADD.h5 weights/
```

**Step 4 — Verify the file is in the right place before running:**

```bash
ls -lh weights/
# Should show:
# -rw-r--r-- 1 user user 51M  phi0linemod_best_ADD.h5
```

> **Colab users:** Upload the weights file to your Google Drive, then mount Drive and copy it into the EfficientPose folder. Weights are too large to upload directly to a Colab session each time.

In [ ]:
# EfficientPose inference pattern (illustrative — requires TF1 environment)
EFFICIENTPOSE_INFERENCE = '''
# efficientpose_demo.py — run inside TF1 environment
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use first GPU

import cv2
import numpy as np
import tensorflow as tf

# ---- Model loading ----
from model import build_EfficientPose
from utils.anchors import anchors_for_shape
from utils.visualization import draw_detections

PHI = 0  # EfficientPose-Phi=0 (fastest)
WEIGHTS = "weights/phi0linemod_best_ADD.h5"
CLASS_NAME = "cup"
SCORE_THRESHOLD = 0.5

# Camera intrinsics (LineMOD default — replace with your calibration!)
CAMERA_MATRIX = np.array([
    [572.4114, 0,       325.2611],
    [0,        573.5704, 242.0490],
    [0,        0,        1      ]
], dtype=np.float64)

# Build model
model, _ = build_EfficientPose(PHI, num_classes=1, score_threshold=SCORE_THRESHOLD)
model.load_weights(WEIGHTS, by_name=True)

# Input size for Phi=0
input_size = 512

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Preprocess
    h, w = frame.shape[:2]
    scale = input_size / max(h, w)
    resized = cv2.resize(frame, (int(w*scale), int(h*scale)))
    padded = np.zeros((input_size, input_size, 3), dtype=np.float32)
    padded[:resized.shape[0], :resized.shape[1]] = resized.astype(np.float32)
    padded = (padded - [103.939, 116.779, 123.68])  # ImageNet normalization
    
    # Inference
    input_batch = np.expand_dims(padded, 0)
    outputs = model.predict_on_batch([input_batch, np.expand_dims(CAMERA_MATRIX, 0)])
    
    # Unpack outputs: boxes, scores, labels, rotation, translation
    boxes, scores, labels, rotation, translation = outputs
    
    # Post-process and draw
    for i, (box, score, label, rot, trans) in enumerate(
            zip(boxes[0], scores[0], labels[0], rotation[0], translation[0])):
        if score < SCORE_THRESHOLD:
            break
        
        # Convert box back to original scale
        x1, y1, x2, y2 = (box / scale).astype(int)
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        
        # rvec, tvec
        rvec = rot.reshape(3, 1)
        tvec = trans.reshape(3, 1)
        
        cv2.putText(frame, f"{CLASS_NAME} | d={float(tvec[2])*100:.0f}cm",
                    (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        
        # Draw axes
        dist_coeffs = np.zeros(5)
        cv2.drawFrameAxes(frame, CAMERA_MATRIX, dist_coeffs,
                          rvec, tvec, 0.05)
    
    cv2.imshow("EfficientPose", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
'''

print('EfficientPose inference script:')
print('Requires: Python 3.7, TF 1.14, CUDA 10.0, EfficientPose repo')
print('='*60)
print(EFFICIENTPOSE_INFERENCE)

## 4. Understanding the Output

EfficientPose returns 5 arrays per batch:

```python
boxes, scores, labels, rotation, translation = model.predict_on_batch([image, camera_matrix])
```

| Output | Shape | Meaning |
|---|---|---|
| `boxes` | `(B, N, 4)` | 2D bounding boxes [x1,y1,x2,y2] |
| `scores` | `(B, N)` | Detection confidence [0,1] |
| `labels` | `(B, N)` | Class index |
| `rotation` | `(B, N, 3)` | Rodrigues rotation vector |
| `translation` | `(B, N, 3)` | Translation vector in meters |

Where B = batch size, N = max detections (sorted by score).

**Key difference from classical methods:**  
You do NOT need `cv2.solvePnP` — the network directly outputs `rvec` and `tvec`.

In [ ]:
# Simulate EfficientPose output and show how to use it
def simulate_efficientpose_output(n_objects=2):
    """Simulate EfficientPose model.predict_on_batch() output."""
    np.random.seed(42)
    N = 100  # max detections
    B = 1    # batch size
    
    boxes       = np.zeros((B, N, 4))
    scores      = np.zeros((B, N))
    labels      = np.zeros((B, N), dtype=int)
    rotation    = np.zeros((B, N, 3))
    translation = np.zeros((B, N, 3))
    
    # Fill in n_objects detections (sorted by score, highest first)
    obj_configs = [
        {'score': 0.92, 'box': [150, 80, 350, 320], 'label': 0,
         'rvec': [0.1, 0.3, -0.2], 'tvec': [0.05, -0.02, 0.45]},
        {'score': 0.71, 'box': [400, 100, 600, 380], 'label': 0,
         'rvec': [-0.2, 0.1, 0.4], 'tvec': [-0.08, 0.01, 0.62]},
    ]
    
    for i, cfg in enumerate(obj_configs[:n_objects]):
        boxes[0, i]       = cfg['box']
        scores[0, i]      = cfg['score']
        labels[0, i]      = cfg['label']
        rotation[0, i]    = cfg['rvec']
        translation[0, i] = cfg['tvec']
    
    return boxes, scores, labels, rotation, translation

boxes, scores, labels, rotation, translation = simulate_efficientpose_output()

SCORE_THRESHOLD = 0.5
CLASS_NAMES = {0: 'cup', 1: 'ape', 2: 'driller'}  # LineMOD classes

print('=== EfficientPose Simulated Output ===')
print()

for i in range(boxes.shape[1]):
    score = scores[0, i]
    if score < SCORE_THRESHOLD:
        break
    
    box   = boxes[0, i]
    label = labels[0, i]
    rvec  = rotation[0, i]
    tvec  = translation[0, i]
    
    R, _ = cv2.Rodrigues(rvec)
    dist  = np.linalg.norm(tvec)
    
    print(f'Detection {i}: {CLASS_NAMES[label]}')
    print(f'  Score:       {score:.2f}')
    print(f'  Box [x1,y1,x2,y2]: {box.astype(int)}')
    print(f'  rvec:        {rvec.round(4)}')
    print(f'  tvec:        {tvec.round(4)} m')
    print(f'  Distance:    {dist*100:.1f} cm')
    print(f'  Rotation angle: {np.degrees(np.linalg.norm(rvec)):.1f}°')
    print()

In [ ]:
# Visualize simulated EfficientPose detections
K = np.array([[572.4, 0, 325.3],[0, 573.6, 242.0],[0,0,1]], dtype=np.float64)
dist_coeffs = np.zeros(5)

# Create synthetic scene
frame = np.ones((480, 640, 3), dtype=np.uint8) * 210
cv2.putText(frame, 'EfficientPose Simulation', (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (80,80,80), 2)

colors = [(0,220,0), (0,100,255)]

for i in range(boxes.shape[1]):
    score = scores[0, i]
    if score < SCORE_THRESHOLD:
        break
    
    box   = boxes[0, i].astype(int)
    label = labels[0, i]
    rvec  = rotation[0, i].reshape(3, 1)
    tvec  = translation[0, i].reshape(3, 1)
    color = colors[i % len(colors)]
    
    # Draw 2D bounding box
    cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), color, 2)
    cv2.putText(frame,
                f'{CLASS_NAMES[label]} {score:.0%} | d={np.linalg.norm(tvec)*100:.0f}cm',
                (box[0], box[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    
    # Draw 3D pose axes
    cv2.drawFrameAxes(frame, K, dist_coeffs, rvec, tvec, 0.04)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title('EfficientPose output — bounding box + 6D pose axes (no PnP needed!)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

## Recap

| Concept | Key Point |
|---|---|
| EfficientPose | 4-head network: class + box + rotation + translation |
| Backbone | EfficientNet with compound scaling |
| Phi (Φ) variants | 0=fastest (32 FPS), 5=most accurate (6 FPS) |
| Output | `rotation` (rvec) + `translation` (tvec) directly — no PnP |
| Trained on | LineMOD dataset (13 objects) |
| ADD metric | Correct if mean point distance < 10% of object diameter |
| Environment | TF 1.14 + Python 3.7 — Docker recommended |
| Camera matrix | Must match your camera — use your calibration from NB 07! |

**Next:** FoundationPose and FreeZe — zero-shot pose on **any** object.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Implement score filtering
# ============================================================
# Using the simulated output (boxes, scores, labels, rotation, translation):
# Write a function that returns only detections above a given threshold.
# Test with thresholds 0.5, 0.8, 0.95.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def filter_detections(boxes, scores, labels, rotation, translation, threshold=0.5):
#     mask = scores[0] >= threshold
#     return {
#         'boxes':       boxes[0][mask],
#         'scores':      scores[0][mask],
#         'labels':      labels[0][mask],
#         'rotation':    rotation[0][mask],
#         'translation': translation[0][mask],
#     }
#
# for thresh in [0.5, 0.8, 0.95]:
#     det = filter_detections(boxes, scores, labels, rotation, translation, thresh)
#     print(f'Threshold={thresh}: {len(det["scores"])} detections')

In [ ]:
# ============================================================
# EXERCISE 2: Implement the ADD metric comparison
# ============================================================
# Given:
#   R_gt = np.eye(3), t_gt = [0.05, -0.02, 0.45] (first detection GT)
# Compute ADD for the first EfficientPose detection.
# Use the model_pts generated in section 2.
# Is the detection considered correct (ADD-0.1d)?
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# R_gt = np.eye(3)
# t_gt = np.array([0.05, -0.02, 0.45])
# 
# rvec_pred = rotation[0, 0]
# t_pred = translation[0, 0]
# R_pred, _ = cv2.Rodrigues(rvec_pred)
# 
# add, correct, diameter = compute_ADD(model_pts, R_pred, t_pred, R_gt, t_gt)
# print(f'ADD: {add*100:.2f}cm')
# print(f'Object diameter: {diameter*100:.2f}cm')
# print(f'Threshold (10%): {diameter*10:.2f}cm')
# print(f'Correct: {"✅" if correct else "❌"}')